## Final Pipeline

###### Access D:\Aamir Gulzar\KSA_project2\dataset\patch_data
###### Iterate through all subfolders and which all have names like TCGA-3L-AA1B_nonMSIH and extract patch level features from them
###### Skip the patches whose name is not in final_merged_without_white.csv
###### patch features are saved in h5, save that h5 in a folder in D:\Aamir Gulzar\KSA_project2\dataset\features\TCGA-3L-AA1B_nonMSIH\patch_features
###### h5 should have name like TCGA-3L-AA1B_nonMSIH_patch_features
###### Extract slide embedding and save that also in D:\Aamir Gulzar\KSA_project2\dataset\features\TCGA-3L-AA1B_nonMSIH\slide_features

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
import os
import glob
import re
import h5py
import pandas as pd
import numpy as np
import torch
from PIL import Image
from typing import List, Tuple, Dict, Any, Union
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel
from huggingface_hub import login

class TITANFeatureExtractor:
    def __init__(self, token: str = os.environ.get("HF_TOKEN", "")):
        """
        Initialize the TITAN feature extractor.
        """
        if token:
            login(token=token)
        
        print("Loading TITAN model...")
        self.titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
        print("Model loaded. Initializing CONCH...")
        self.conch, self.eval_transform = self.titan.return_conch()
        
        # Move model to GPU if available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")
        self.conch = self.conch.to(self.device)
        self.titan = self.titan.to(self.device)

        self.conch.eval()
        self.titan.eval()

def extract_coords_from_filename(filename: str) -> Tuple[int, int]:
    """
    Extract x, y coordinates from patch filename.
    Expected format: ..._x(\d+)_y(\d+)...
    """
    basename = os.path.splitext(os.path.basename(filename))[0]
    pattern = r'_x(\d+)_y(\d+)'
    match = re.search(pattern, basename)
    if not match:
        raise ValueError(f"Could not extract coordinates from filename: {filename}")
    return int(match.group(1)), int(match.group(2))

def sort_by_coords(image_paths: List[str]) -> List[str]:
    """Sort image paths by coordinates extracted from filenames."""
    def get_xy(path):
        try:
            return extract_coords_from_filename(path)
        except ValueError:
            return (float('inf'), float('inf'))
    return sorted(image_paths, key=get_xy)

def load_image_batch(image_paths: List[str], start_idx: int, batch_size: int, transform) -> torch.Tensor:
    """Load and transform a batch of images efficiently using threading."""
    end_idx = min(start_idx + batch_size, len(image_paths))
    batch_paths = image_paths[start_idx:end_idx]
    
    def load_single_image(path):
        try:
            img = Image.open(path).convert("RGB")
            return transform(img)
        except Exception as e:
            print(f"Warning: Failed to load {path}: {e}")
            return None
    
    with ThreadPoolExecutor(max_workers=min(8, len(batch_paths))) as executor:
        tensors = list(executor.map(load_single_image, batch_paths))
    
    valid_tensors = [t for t in tensors if t is not None]
    if not valid_tensors:
        return torch.empty(0)
    
    return torch.stack(valid_tensors)

def create_h5_from_features(
    features: np.ndarray,
    image_paths: List[str],
    output_dir: str,
    slide_name: str,
    patch_size_level0: int = 1024,
    additional_attributes: Dict[str, Any] = None
) -> str:
    """Create H5 file with features and coordinates."""
    if len(features) != len(image_paths):
        raise ValueError(f"Features length ({len(features)}) must match image_paths length ({len(image_paths)})")
    
    def extract_coords_worker(img_path):
        try:
            x, y = extract_coords_from_filename(img_path)
            return [x, y]
        except ValueError as e:
            print(f"Warning: {e}")
            return None
    
    with ThreadPoolExecutor(max_workers=8) as executor:
        coords_list = list(executor.map(extract_coords_worker, image_paths))
    
    coords = np.array([c for c in coords_list if c is not None], dtype=np.int64)
    
    h5_filename = f"{slide_name}_patch_features.h5"
    output_path = os.path.join(output_dir, h5_filename)
    os.makedirs(output_dir, exist_ok=True)
    
    # Add batch dimensions for TITAN compatibility (1, N, D)
    features_batch = features[np.newaxis, :]
    coords_batch = coords[np.newaxis, :]
    
    with h5py.File(output_path, 'w') as h5f:
        h5f.create_dataset('features', data=features_batch.astype(np.float32), dtype=np.float32)
        coords_dataset = h5f.create_dataset('coords', data=coords_batch.astype(np.int64), dtype=np.int64)
        
        attrs = {
            'contour_fn': 'four_pt',
            'custom_downsample': 2.0,
            'downsample': np.array([1., 1.]),
            'name': slide_name,
            'patch_level': 0,
            'patch_size': patch_size_level0,
            'patch_size_level0': patch_size_level0,
            'step_size': patch_size_level0,
            'use_padding': True
        }
        if additional_attributes:
            attrs.update(additional_attributes)
        
        for key, value in attrs.items():
            coords_dataset.attrs[key] = value
            
    return output_path

def run_full_pipeline(
    patch_data_dir: str,
    extractor: TITANFeatureExtractor,
    slide_meta_csv: str,
    non_white_csv: str,
    output_root: str,
    batch_size: int = 64
):
    """
    Main pipeline to process all slide folders and generate embeddings.
    """
    print(f"Loading metadata from {slide_meta_csv}...")
    slide_meta_df = pd.read_csv(slide_meta_csv)
    
    print(f"Loading non-white patch list from {non_white_csv}...")
    non_white_df = pd.read_csv(non_white_csv)
    valid_patch_names = set(non_white_df['patch_name'].astype(str))
    
    # Create output directories
    h5_dir = os.path.join("D:\Aamir Gulzar\KSA_project2\paip_data\slide_h5_files", "TITAN")
    emb_dir = os.path.join(output_root, "Conch1_5")
    os.makedirs(h5_dir, exist_ok=True)
    os.makedirs(emb_dir, exist_ok=True)
    
    # Iterating through slide directories
    slide_dirs = [d for d in glob.glob(os.path.join(patch_data_dir, "*")) if os.path.isdir(d)]
    print(f"Found {len(slide_dirs)} slide folders to process.")
    
    for slide_dir in slide_dirs:
        slide_name = os.path.basename(slide_dir)
        print(f"\nProcessing Slide: {slide_name}")
        
        # Check if already processed
        pt_path = os.path.join(emb_dir, f"{slide_name}.pt")
        if os.path.exists(pt_path):
            print(f"Skipping {slide_name}, embedding already exists.")
            continue
            
        # 1. Get Metadata
        meta_row = slide_meta_df[slide_meta_df["renamed"].str.lower() == slide_name.lower()]
        if meta_row.empty:
            print(f"Error: No metadata found for {slide_name} in {slide_meta_csv}. Skipping.")
            continue
            
        obj_power = int(meta_row.iloc[0]["Objective Power"])
        patch_size_level0 = 1024 if obj_power == 40 else 512 if obj_power == 20 else None
        if patch_size_level0 is None:
            print(f"Error: Unsupported objective power {obj_power} for {slide_name}. Skipping.")
            continue
        print(f"Objective Power: {obj_power}X -> Patch Size: {patch_size_level0}")
        
        # 2. Filter Patches
        all_patches = glob.glob(os.path.join(slide_dir, "*.png"))
        patch_paths = [p for p in all_patches if os.path.basename(p) in valid_patch_names]
        
        if not patch_paths:
            print(f"Warning: No valid patches found for {slide_name} after filtering with non_white_csv. Skipping.")
            continue
            
        print(f"Found {len(patch_paths)} patches to extract features from.")
        patch_paths = sort_by_coords(patch_paths)
        
        # 3. Extract Features
        all_features = []
        with torch.inference_mode():
            for i in range(0, len(patch_paths), batch_size):
                print(f"  Batch {i//batch_size + 1}/{(len(patch_paths)-1)//batch_size + 1}...", end="\r")
                batch_tensor = load_image_batch(patch_paths, i, batch_size, extractor.eval_transform)
                if batch_tensor.nelement() == 0:
                    continue
                
                batch_tensor = batch_tensor.to(extractor.device)
                features = extractor.conch(batch_tensor)
                all_features.append(features.cpu().numpy())
        
        print("\n  Collating features...")
        features_np = np.concatenate(all_features, axis=0)
        
        # 4. Save to H5 (Optional but good for record)
        h5_path = create_h5_from_features(
            features=features_np,
            image_paths=patch_paths,
            output_dir=h5_dir,
            slide_name=slide_name,
            patch_size_level0=patch_size_level0
        )
        print(f"  Saved patch features to {h5_path}")
        
        # 5. Generate Slide Embedding
        print("  Encoding slide embedding...")

        coords = np.array(
            [extract_coords_from_filename(p) for p in patch_paths],
            dtype=np.int64
        )

        features_torch = torch.from_numpy(features_np).unsqueeze(0).to(extractor.device)
        coords_torch = torch.from_numpy(coords).unsqueeze(0).to(extractor.device)

        with torch.inference_mode():
            slide_emb = extractor.titan.encode_slide_from_patch_features(
                features_torch,
                coords_torch,
                patch_size_level0
            )
            
        # Save as .pt
        torch.save(slide_emb.cpu(), pt_path)
        print(f"  Successfully saved slide embedding to {pt_path}")

if __name__ == "__main__":
    # Update these paths
    PATCH_DATA_DIR = r"D:\Aamir Gulzar\KSA_project2\paip_data\patch_data"
    SLIDE_META_CSV = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\data_preprocessing\slides_metadata\paip_slide_metadata_filtered.csv"
    NON_WHITE_CSV = r"D:\Aamir Gulzar\KSA_project2\paip_data\final_merged_without_white.csv"
    OUTPUT_ROOT = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\TITAN"
    
    # Initialize Extractor (loads TITAN and CONCH1_5)
    extractor = TITANFeatureExtractor()
    
    # Run the pipeline
    run_full_pipeline(
        patch_data_dir=PATCH_DATA_DIR,
        extractor=extractor,
        slide_meta_csv=SLIDE_META_CSV,
        non_white_csv=NON_WHITE_CSV,
        output_root=OUTPUT_ROOT,
        batch_size=64 # Adjust based on GPU memory
    )


Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
Loading TITAN model...
Model loaded. Initializing CONCH...
Using device: cuda
Loading metadata from D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\data_preprocessing\slides_metadata\paip_slide_metadata_filtered.csv...
Loading non-white patch list from D:\Aamir Gulzar\KSA_project2\paip_data\final_merged_without_white.csv...
Found 73 slide folders to process.

Processing Slide: training_data_01_MSIH
Skipping training_data_01_MSIH, embedding already exists.

Processing Slide: training_data_02_nonMSIH
Skipping training_data_02_nonMSIH, embedding already exists.

Processing Slide: training_data_03_nonMSIH
Skipping training_data_03_nonMSIH, embedding already exists.

Processing Slide: training_data_04_nonMSIH
Skipping t

In [9]:
h5_path = r"D:\Aamir Gulzar\KSA_project2\dataset\Features\TCGA-3L-AA1B_nonMSIH\patch_features\TCGA-3L-AA1B_nonMSIH_patch_features.h5"

# Open and read the features
with h5py.File(h5_path, 'r') as h5_file:
    features = h5_file['features'][:]  # shape: (N_patches, feature_dim)

print(features.shape)

features = features[0]  # shape becomes (1878, 768)

# Compute variance across all patches for each feature dimension
variances = np.var(features, axis=0)

# Compute and print mean variance
mean_variance = np.mean(variances)
print("Mean variance of patch features:", mean_variance)

(1, 1878, 768)
Mean variance of patch features: 0.42321482


## Analyse .pt files

In [10]:
import torch
import numpy as np

def analyze_pt_file(pt_path):
    print(f"\nAnalyzing file: {pt_path}")

    data = torch.load(pt_path, map_location="cpu")

    print("\n===== Basic Info =====")

    if isinstance(data, torch.Tensor):
        tensor = data
    elif isinstance(data, dict):
        print("File contains a dictionary. Keys:", list(data.keys()))
        tensor = list(data.values())[0]
    else:
        raise ValueError("Unsupported PT structure")

    print("Type:", type(tensor))
    print("Shape:", tensor.shape)
    print("Dtype:", tensor.dtype)
    print("Device:", tensor.device)

    arr = tensor.numpy()

    print("\n===== Value Statistics =====")
    print("Min:", np.min(arr))
    print("Max:", np.max(arr))
    print("Mean:", np.mean(arr))
    print("Std:", np.std(arr))

    print("\n===== Integrity Checks =====")

    nan_count = np.isnan(arr).sum()
    inf_count = np.isinf(arr).sum()

    print("NaN values:", nan_count)
    print("Inf values:", inf_count)

    if nan_count == 0 and inf_count == 0:
        print("✔ No NaN or Inf values detected")
    else:
        print("⚠ Potential numerical issues detected")

    print("\n===== First 10 Values =====")
    print(arr.flatten()[:10])

    print("\n===== L2 Norm =====")
    norm = np.linalg.norm(arr)
    print(norm)


if __name__ == "__main__":
    pt_file = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\TITAN\Conch1_5\training_data_04_nonMSIH.pt"

    analyze_pt_file(pt_file)


Analyzing file: D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\TITAN\Conch1_5\training_data_04_nonMSIH.pt

===== Basic Info =====
Type: <class 'torch.Tensor'>
Shape: torch.Size([1, 768])
Dtype: torch.float32
Device: cpu

===== Value Statistics =====
Min: -2.033909
Max: 1.8545138
Mean: -0.0064843087
Std: 0.5751785

===== Integrity Checks =====
NaN values: 0
Inf values: 0
✔ No NaN or Inf values detected

===== First 10 Values =====
[-0.08206127  0.09819509 -0.13292716 -0.03987861  0.22279404  0.33500066
 -0.18833143 -0.9480904  -0.13916363 -0.51063824]

===== L2 Norm =====
15.940826
